# Embedding-based LOCo split + chemical (Tanimoto) homology audit

Run cells top to bottom. **Stop after Cell 2 (the sweep)** and read its output before running Cell 3 -- the threshold has to come from the sweep, not a guess (0.3 collapsed everything into 1-2 clusters twice already).

Why Tanimoto/RDKit instead of MMSeqs: this dataset only has SMILES (`smiles_canonical`), no amino-acid sequence column -- MMSeqs needs FASTA and can't run on molecule strings. Tanimoto similarity over Morgan (ECFP) fingerprints is the molecule-space equivalent of sequence identity.

Why a k-NN graph instead of a radius graph for clustering: a radius graph's size depends on the data (how many points actually fall within the threshold), so a bad threshold can make it explode to ~n^2 edges regardless of chunking -- that's what caused the earlier RAM crashes. A k-NN graph is capped at `n * K_NEIGHBORS` edges no matter how dense the embedding space is.

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
!pip install rdkit


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.4/37.4 MB 41.1 MB/s eta 0:00:00


## Cell 1: Config + load data

In [5]:
import numpy as np
import pandas as pd
from rdkit import Chem
from rdkit.Chem import DataStructs, rdFingerprintGenerator
from scipy.sparse.csgraph import connected_components
from sklearn.neighbors import NearestNeighbors

pd.read_csv("/content/drive/MyDrive/Dataset/phase2_joint_clusters.csv").to_parquet("df_dedup.parquet")

DF_PATH = "df_dedup.parquet"
EMBEDDINGS_PATH = "/content/drive/MyDrive/Dataset/phase2_embeddings.npy"
ID_COL = "peptide_uid"
SMILES_COL = "smiles_canonical"
K_NEIGHBORS = 15                         # hard cap on graph edges per point -- tune freely, doesn't affect memory safety
TANIMOTO_CUTOFF = 0.85                   # similarity treated as "same/near-duplicate molecule"
MORGAN_RADIUS, MORGAN_NBITS = 2, 2048    # ECFP4-equivalent fingerprint settings
TRAIN_FRAC, VAL_FRAC, TEST_FRAC = 0.8, 0.1, 0.1
OUTPUT_DF_PATH = "df_with_splits.parquet"
OUTPUT_LEAKS_PATH = "leaked_pairs_audit.csv"

df = pd.read_parquet(DF_PATH)
embeddings = np.load(EMBEDDINGS_PATH)
assert len(df) == len(embeddings), (
    f"df ({len(df)} rows) and embeddings ({len(embeddings)} rows) must be row-aligned"
)
print(f"Loaded {len(df)} peptides, embeddings shape {embeddings.shape}")

Loaded 34971 peptides, embeddings shape (34971, 1024)


## Cell 2: distance distribution + threshold sweep

Read the printed percentiles and sweep table before moving to Cell 3. Pick a threshold where `n_clusters` is in the dozens-to-low-hundreds and `largest_cluster_frac` is nowhere near 1.0.

In [6]:
def summarize_neighbor_distances(embeddings: np.ndarray, k: int = K_NEIGHBORS) -> np.ndarray:
    nn = NearestNeighbors(n_neighbors=k, metric="cosine")
    nn.fit(embeddings)
    distances = nn.kneighbors_graph(embeddings, mode="distance").data
    print(f"Cosine distance percentiles across all {len(distances)} k-NN edges (k={k}):")
    for p in [1, 5, 10, 25, 50, 75, 90]:
        print(f"  {p:>2}th percentile: {np.percentile(distances, p):.4f}")
    return distances


def cluster_embeddings(
    embeddings: np.ndarray, distance_threshold: float, k_neighbors: int = K_NEIGHBORS
) -> np.ndarray:
    nn = NearestNeighbors(n_neighbors=k_neighbors, metric="cosine")
    nn.fit(embeddings)
    dist_graph = nn.kneighbors_graph(embeddings, mode="distance")   # bounded: n * k_neighbors edges, max
    dist_graph.data = (dist_graph.data <= distance_threshold).astype(np.float32)
    dist_graph.eliminate_zeros()                                    # drop edges beyond the threshold
    _, labels = connected_components(dist_graph, directed=False)
    return labels


def sweep_thresholds(embeddings: np.ndarray, thresholds: list[float]) -> None:
    for t in thresholds:
        labels = cluster_embeddings(embeddings, t)
        sizes = pd.Series(labels).value_counts()
        print(f"threshold={t:.4f}  n_clusters={len(sizes)}  "
              f"largest_cluster_frac={sizes.iloc[0] / len(labels):.3f}")


neighbor_distances = summarize_neighbor_distances(embeddings)
sweep_candidates = [np.percentile(neighbor_distances, p) for p in [1, 2, 5, 10, 15, 20, 30, 50]]
sweep_thresholds(embeddings, sweep_candidates)

Cosine distance percentiles across all 524565 k-NN edges (k=15):
   1th percentile: 0.0000
   5th percentile: 0.0000
  10th percentile: 0.0010
  25th percentile: 0.0067
  50th percentile: 0.0188
  75th percentile: 0.0323
  90th percentile: 0.0489
threshold=0.0000  n_clusters=33752  largest_cluster_frac=0.001
threshold=0.0000  n_clusters=33752  largest_cluster_frac=0.001
threshold=0.0000  n_clusters=33585  largest_cluster_frac=0.001
threshold=0.0010  n_clusters=29907  largest_cluster_frac=0.002
threshold=0.0030  n_clusters=24801  largest_cluster_frac=0.003
threshold=0.0048  n_clusters=21426  largest_cluster_frac=0.006
threshold=0.0088  n_clusters=16987  largest_cluster_frac=0.029
threshold=0.0188  n_clusters=9623  largest_cluster_frac=0.421


In [7]:
sweep_thresholds(embeddings, [0.009, 0.010, 0.011, 0.012, 0.013, 0.014, 0.015, 0.016])

threshold=0.0090  n_clusters=16819  largest_cluster_frac=0.031
threshold=0.0100  n_clusters=15996  largest_cluster_frac=0.042
threshold=0.0110  n_clusters=15235  largest_cluster_frac=0.072
threshold=0.0120  n_clusters=14417  largest_cluster_frac=0.106
threshold=0.0130  n_clusters=13677  largest_cluster_frac=0.122
threshold=0.0140  n_clusters=12934  largest_cluster_frac=0.143
threshold=0.0150  n_clusters=12268  largest_cluster_frac=0.166
threshold=0.0160  n_clusters=11528  largest_cluster_frac=0.215


In [8]:
labels = cluster_embeddings(embeddings, 0.0100)
dupe_groups = df.assign(_c=labels).groupby("smiles_canonical")["_c"].nunique()
split_dupes = (dupe_groups > 1).sum()
print(f"{split_dupes} identical-SMILES groups got split across clusters (want 0)")

0 identical-SMILES groups got split across clusters (want 0)


## Cell 3: cluster at chosen threshold, bin-pack into train/val/test

**Set `DISTANCE_THRESHOLD` below from the Cell 2 output before running this.**

In [9]:
DISTANCE_THRESHOLD = 0.0100  # <-- set this from the Cell 2 sweep output, not a guess

def assign_clusters_to_splits(
    cluster_labels: np.ndarray,
    train_frac: float = 0.8,
    val_frac: float = 0.1,
    test_frac: float = 0.1,
) -> np.ndarray:
    assert abs(train_frac + val_frac + test_frac - 1.0) < 1e-6

    unique, counts = np.unique(cluster_labels, return_counts=True)
    order = np.argsort(-counts)  # largest cluster first

    targets = {"train": train_frac, "val": val_frac, "test": test_frac}
    split_counts = {"train": 0, "val": 0, "test": 0}
    total = len(cluster_labels)
    cluster_to_split = {}

    for idx in order:
        cluster_id, size = unique[idx], counts[idx]
        deficits = {
            s: targets[s] - (split_counts[s] / total if total else 0)
            for s in split_counts
        }
        chosen = max(deficits, key=deficits.get)
        split_counts[chosen] += size
        cluster_to_split[cluster_id] = chosen

    return np.array([cluster_to_split[c] for c in cluster_labels])


labels = cluster_embeddings(embeddings, DISTANCE_THRESHOLD)
splits = assign_clusters_to_splits(labels, TRAIN_FRAC, VAL_FRAC, TEST_FRAC)

df["embedding_cluster"] = labels
df["split"] = splits

print(f"{len(np.unique(labels))} clusters from {len(df)} peptides "
      f"(distance_threshold={DISTANCE_THRESHOLD})")
print(df["split"].value_counts(normalize=True).round(3))

15996 clusters from 34971 peptides (distance_threshold=0.01)
split
train    0.8
test     0.1
val      0.1
Name: proportion, dtype: float64


## Cell 4: Tanimoto homology audit (optional)

Checks val/test peptides against the full set (not full all-pairs -- full all-pairs Tanimoto at n=35K is the same O(n^2) trap that broke earlier steps). Skip this cell if you just want the split without the audit.

In [10]:
def compute_fingerprints(smiles_list: list[str], radius: int, n_bits: int):
    generator = rdFingerprintGenerator.GetMorganGenerator(radius=radius, fpSize=n_bits)
    fps, valid_mask = [], []
    for s in smiles_list:
        mol = Chem.MolFromSmiles(s) if isinstance(s, str) else None
        if mol is None:
            fps.append(None)
            valid_mask.append(False)
        else:
            fps.append(generator.GetFingerprint(mol))
            valid_mask.append(True)
    return fps, np.array(valid_mask)


def chemical_homology_audit(
    df: pd.DataFrame,
    smiles_col: str,
    id_col: str,
    split_col: str,
    similarity_cutoff: float,
    radius: int,
    n_bits: int,
) -> pd.DataFrame:
    fps, valid_mask = compute_fingerprints(df[smiles_col].tolist(), radius, n_bits)
    n_invalid = (~valid_mask).sum()
    if n_invalid:
        print(f"{n_invalid} rows had unparsable SMILES -- excluded from audit.")

    ids = df[id_col].to_numpy()
    split_vals = df[split_col].to_numpy()
    valid_idx = np.where(valid_mask)[0]
    query_idx = valid_idx[split_vals[valid_idx] != "train"]  # only val/test as queries

    print(f"Checking {len(query_idx)} val/test peptides against "
          f"{len(valid_idx)} total -- not full {len(valid_idx)}x{len(valid_idx)} all-pairs.")

    leaked = []
    seen_pairs = set()
    for qi in query_idx:
        sims = DataStructs.BulkTanimotoSimilarity(fps[qi], [fps[j] for j in valid_idx])
        for pos, sim in enumerate(sims):
            j = valid_idx[pos]
            if j == qi or sim < similarity_cutoff or split_vals[j] == split_vals[qi]:
                continue
            pair_key = tuple(sorted([ids[qi], ids[j]]))
            if pair_key in seen_pairs:
                continue
            seen_pairs.add(pair_key)
            leaked.append((ids[qi], ids[j], sim, split_vals[qi], split_vals[j]))

    return pd.DataFrame(
        leaked, columns=["query_uid", "target_uid", "tanimoto", "split_query", "split_target"]
    )


leaked_pairs = chemical_homology_audit(
    df, SMILES_COL, ID_COL, "split", TANIMOTO_CUTOFF, MORGAN_RADIUS, MORGAN_NBITS
)

print(f"{len(leaked_pairs)} pairs with Tanimoto >= {TANIMOTO_CUTOFF} "
      f"split across different sets.")
if len(leaked_pairs):
    print(leaked_pairs.sort_values("tanimoto", ascending=False).head(10).to_string(index=False))

Checking 6994 val/test peptides against 34971 total -- not full 34971x34971 all-pairs.
83575 pairs with Tanimoto >= 0.85 split across different sets.
                       query_uid                       target_uid  tanimoto split_query split_target
2d7f2c385227c49a02b79efd95b60b1e 6690ef6aca298d9c6fc9930a6d283546       1.0        test        train
ed7ab012a0e44216c34845089edc5bbc ea1c73cae2f24beea47c7e90d840c767       1.0         val        train
ecdf81a41447dca02938676e01590fc3 4800ed3a4a62f9ad5048e43eb25487e0       1.0        test        train
a4462e70c9dc5fcc382b5424c1647b96 da26aaccbb220128ffd67eab9bb5794a       1.0        test        train
ecdf81a41447dca02938676e01590fc3 70ec25beaf3b902807c1db2664163684       1.0        test        train
3bf47caa2a2e42b353bdfb97d605d16b 8e6f6c3526f9491ef3939e5125c21725       1.0        test        train
aa164330e8d21a4b82de32cb12ea7bef db8ce39d980a259700d689ec87f89b50       1.0        test        train
aa164330e8d21a4b82de32cb12ea7bef bd891d326

In [11]:
smiles_by_uid = dict(zip(df["peptide_uid"], df["smiles_canonical"]))
perfect = leaked_pairs[leaked_pairs["tanimoto"] == 1.0].copy()
perfect["same_smiles"] = [
    smiles_by_uid[q] == smiles_by_uid[t]
    for q, t in zip(perfect["query_uid"], perfect["target_uid"])
]
print(perfect["same_smiles"].value_counts())

same_smiles
False    1975
Name: count, dtype: int64


In [12]:
n_affected = leaked_pairs["query_uid"].nunique()
print(f"{n_affected} of 6994 val/test peptides ({n_affected/6994:.1%}) have a cross-split near-duplicate")


4156 of 6994 val/test peptides (59.4%) have a cross-split near-duplicate


## Cell 5: save outputs

In [13]:
df.to_parquet(OUTPUT_DF_PATH)
print(f"Saved {OUTPUT_DF_PATH}")

if "leaked_pairs" in dir():
    leaked_pairs.to_csv(OUTPUT_LEAKS_PATH, index=False)
    print(f"Saved {OUTPUT_LEAKS_PATH}")
else:
    print("Cell 4 (homology audit) wasn't run -- skipped saving leaked_pairs_audit.csv")

Saved df_with_splits.parquet
Saved leaked_pairs_audit.csv
